# Step 0: Proportional Stratified Sampling of KBR Inventory
## Overview
This notebook implements the initial sampling phase of the KBR Google Books digitization project analysis pipeline. To ensure a representative subset for subsequent analysis, it performs proportional stratified sampling based on collection sections (`M-SLZ` and `M-RP`) after executing an data de-duplication pass.

**Input**: A master inventory CSV file `(0423_all_works_GB.csv`) containing raw, potentially duplicated bibliographic holdings.

**Output**: A verified plain-text list of 1,000 unique internal document identifiers (`Q1_kbr_idn_list.txt`).

**Key Design Decisions**:

* **Data Integrity**: Primary deduplication is enforced strictly on the unique identifier field (`IDN`) before any sampling logic occurs.
  
* **Stratification Ratio**: The target 1,000-sample size maintains the historical proportional distribution between sections (81% for `M-SLZ` and 19% for `M-RP`).
* **Reproducibility**: A fixed pseudo-random number generator seed (`random_state=42`) is applied across all data splits to guarantee identical output streams upon re-execution.

# Importing Environment

In [1]:
import pandas as pd

# Data Ingestion & De-duplication

In [7]:
# 1. Read data
df = pd.read_csv('../data/raw/0423_Q1_list_all_works_GB.csv', encoding='latin1')

# 2. drop duplicates based on the 'IDN' column, keeping only the first occurrence
df_unique = df.drop_duplicates(subset=['IDN'])

# Check if data is loaded correctly (shows the first 5 rows)
df.head()


,ï»¿Project,IDN,Shelf Number,Section,Title/Subtitle/Responsibility,Production,Description,Note,Reasons,Cart,Cart status,Shipment sent date,Shipment received date
0,SA,11247408,"II 11.824, II 16.319 A",M-SLZ,Handbook to the cathedrals of England,Londen / John Murray / 1862-1864,"2 volumes (VIII, 358 ; VII, 325 pagina's) / il...",NaN,NaN,KBR029,Finished,2024/4/10,2024/6/5
1,SA,11247519,II 16.323 A,M-SLZ,Der grosse Wolfdieterich / von Adolf Holtzmann,Heidelberg / [Herausgeber nicht ermittelbar] /...,"CI, 364 Seiten / 23 cm",NaN,NaN,KBR038,Finished,2024/4/10,2024/6/5
2,SA,11247699,II 16.328 A,M-SLZ,Nothern Mythology / comprising the principal ...,Londen / Edward Lumley / 1851,"3 volumes (XIII, 307; XXVIII, 284 ; X, 340, 20...",NaN,NaN,KBR038,Finished,2024/4/10,2024/6/5
3,SA,11247732,II 16.329 A,M-SLZ,James Brindley and the early engineers / door ...,Londen / John Murray / 1864,"XIII, 320, 32 pagina's / illustraties / 19 cm",NaN,NaN,KBR038,Finished,2024/4/10,2024/6/5
4,SA,11247765,II 16.330 A,M-SLZ,A genealogical and heraldic Dictionary of the ...,Londen / [Uitgever niet geÃ¯dentificeerd] / 1865,"XLVII, 1323, 60 pagina's / illustraties / 25 cm",NaN,NaN,KBR038,Finished,2024/4/10,2024/6/5


# Stratum Segmentation & Distribution Metrics

In [8]:
# 3. Use df_unique for the following filtering steps
df_slz = df_unique[df_unique['Section'] == 'M-SLZ']
df_rp = df_unique[df_unique['Section'] == 'M-RP']

print(f"De-duplicated SLZ stock count: {len(df_slz)}")
print(f"De-duplicated RP stock count: {len(df_rp)}")

De-duplicated SLZ stock count: 56802
De-duplicated RP stock count: 11234


# Proportional Stratified Random Sampling

In [9]:
# Random sampling, random_state=42 ensures the results are reproducible
sample_slz = df_slz.sample(n=810, random_state=42)
sample_rp = df_rp.sample(n=190, random_state=42)

# Merge the two sample sets
final_sample = pd.concat([sample_slz, sample_rp])

# Identifier Extraction & File Serialization

In [ ]:
# Extract the IDN column and convert it to a list
idn_list = final_sample['IDN'].astype(str).tolist()

# Save as a txt file, one IDN per line
with open('../data/raw/Q1_kbr_idn_list.txt', 'w') as f:
    f.write('\n'.join(idn_list))

print("Done! Please check your folder.")

Done! Please check your folder.


# Automated Quality Assurance & Verification Report

In [13]:
# Read the newly generated txt file for verification
with open('../data/raw/Q1_kbr_idn_list.txt', 'r') as f:
    checked_idns = f.read().splitlines()

# 1. Check if the total count is exactly 1000
print(f"--- Verification Report ---")
print(f"Total IDN count: {len(checked_idns)} (Expected: 1000)")

# 2. Check for duplicate IDNs (in case the sampling logic has issues)
unique_count = len(set(checked_idns))
print(f"Total Unique IDN count: {unique_count}")

# 3. Check data format (any strange symbols or scientific notation)
error_idns = [i for i in checked_idns if '+' in i or 'E' in i or '.' in i]
if not error_idns:
    print("Data format check: Passed (no scientific notation or decimal points)")
else:
    print(f"!!! Warning: Found {len(error_idns)} records with potentially incorrect format: {error_idns[:5]}")

# 4. Randomly print the first 3 and last 3 IDs, so you can cross-reference with the original CSV file
print(f"\nFirst 3 IDNs: {checked_idns[:3]}")
print(f"Last 3 IDNs: {checked_idns[-3:]}")


--- Verification Report ---
Total IDN count: 1000 (Expected: 1000)
Total Unique IDN count: 1000
Data format check: Passed (no scientific notation or decimal points)

First 3 IDNs: ['12163581', '11528967', '12021785']
Last 3 IDNs: ['13396259', '11919482', '11795091']
